In [6]:
import os
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# ==========================================
# 👩‍💻 [개별 실습 과제 (TODO) - 권장 시간: 20분]
# ==========================================
print("=== 📝 TODO: 메모리를 활용한 Stateful 연구 노트 작성 에이전트 ===")
print("연구를 진행할 때, 에이전트가 단발성으로 대답하고 끝나는 것이 아니라")
print("대화 기록(Memory)을 유지하며 사용자의 요청에 따라 가상의 '연구 노트'에 내용을 누적 기록하는 시스템을 만드세요.")

# 가상의 연구 노트 (전역 변수로 상태 유지)
# (실제 환경에서는 DB에 저장하겠지만, 실습을 위해 dict를 사용합니다)
research_notebook = {"title": "미정", "content": ""}

# TODO 1: 연구 노트를 수정하는 도구 생성
# 제목(title)이나 추가할 내용(content_to_append)을 받아 research_notebook 딕셔너리를 업데이트하는 도구를 만드세요.
@tool
def update_research_notebook(title: str = None, content_to_append: str = None) -> str:
    """연구 노트의 제목을 설정하거나, 새로운 발견 사항을 노트에 추가 기록할 때 사용하는 도구입니다."""
    global research_notebook
    state = ""
    # 로직: title이 있으면 업데이트, content_to_append가 있으면 기존 content에 문자열 이어붙이기
    if title:
        research_notebook["title"] = title
        state += f"제목이 '{title}'로 설정되었습니다. "
    if content_to_append:
        if research_notebook["content"]:
            research_notebook["content"] += " " + content_to_append
            state += f"새로운 내용이 노트에 추가되었습니다. "
        else:
            research_notebook["content"] = content_to_append
            state += f"노트에 첫 번째 내용이 추가되었습니다. "
    return state.strip()
    # 완료 후 현재 노트의 상태를 문자열로 반환

# TODO 2: 읽기 전용 도구 생성 (현재 노트 상태 확인)
@tool
def read_research_notebook() -> str:
    """현재까지 작성된 연구 노트의 전체 내용을 읽어옵니다."""
    return f"제목: {research_notebook['title']}\n내용: {research_notebook['content']}"

# TODO 3: MemorySaver를 장착한 에이전트 생성
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tools = [update_research_notebook, read_research_notebook]
memory = MemorySaver() # MemorySaver 인스턴스 생성
agent_executor = create_react_agent(llm, tools, checkpointer=memory)

# TODO 4: 3-Turn 대화 시뮬레이션
config = {"configurable": {"thread_id": "lab_session_101"}}
# 턴 1: "오늘 연구 노트 제목을 '양자 컴퓨터 트렌드 분석'으로 설정해줘."
user_input = "오늘 연구 노트 제목을 '양자 컴퓨터 트렌드 분석'으로 설정해줘."
response = agent_executor.invoke(
    {
        "messages": [HumanMessage(content=user_input)]
    }, config=config
)
print("결과:", response["messages"][-1].content)

# 턴 2: "최근 논문에 따르면 양자 오류 정정 기술이 핵심이라고 해. 이 내용을 노트에 기록해둬."
user_input = "최근 논문에 따르면 양자 오류 정정 기술이 핵심이라고 해. 이 내용을 노트에 기록해둬."
response = agent_executor.invoke(
    {
        "messages": [HumanMessage(content=user_input)]
    }, config=config
)
print("결과:", response["messages"][-1].content)

# 턴 3: "지금까지 내 노트에 뭐라고 적혀있는지 원문 그대로 출력해봐."
user_input = "지금까지 내 노트에 뭐라고 적혀있는지 원문 그대로 출력해봐."
response = agent_executor.invoke(
    {
        "messages": [HumanMessage(content=user_input)]
    }, config=config
)
print("결과:", response["messages"][-1].content)
# (에이전트가 thread_id를 통해 상태를 기억하고 도구를 누적해서 사용하는지 확인하세요)

=== 📝 TODO: 메모리를 활용한 Stateful 연구 노트 작성 에이전트 ===
연구를 진행할 때, 에이전트가 단발성으로 대답하고 끝나는 것이 아니라
대화 기록(Memory)을 유지하며 사용자의 요청에 따라 가상의 '연구 노트'에 내용을 누적 기록하는 시스템을 만드세요.


/var/folders/lf/8x594xb50nng1wg8gz583qpc0000gn/T/ipykernel_59957/533385955.py:50: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools, checkpointer=memory)
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


결과: 연구 노트의 제목이 '양자 컴퓨터 트렌드 분석'으로 설정되었습니다. 추가로 기록할 내용이 있으면 말씀해 주세요!


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


결과: 노트에 "최근 논문에 따르면 양자 오류 정정 기술이 핵심이라고 한다."라는 내용이 추가되었습니다. 더 기록할 내용이 있으면 알려주세요!


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


결과: 현재 연구 노트의 내용은 다음과 같습니다:

**제목:** 양자 컴퓨터 트렌드 분석  
**내용:** 최근 논문에 따르면 양자 오류 정정 기술이 핵심이라고 한다. 

더 추가할 내용이나 수정할 사항이 있으면 말씀해 주세요!


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
